# Whole documents with Unlimited-OCR

`baidu/Unlimited-OCR` is the most-downloaded OCR release mlx-vlm supports
(2.3 M) and fits 16 GB at 2.4 GB in 4-bit.

It has two documented modes, and they need different processor geometry:

| mode | prompt | geometry |
|---|---|---|
| single page | `document parsing.` | `gundam` — local crops plus a global view (the default) |
| many pages | `Multi page parsing.` | `base` — `cropping=False`, `image_size=1024` |

In [1]:
from mlx_vlm import apply_chat_template, generate, load

MODEL = "mlx-community/Unlimited-OCR-4bit"
PAGE = "../images/demo_pdf1_page1.png"

model, processor = load(MODEL)

Add pad token = ['<｜▁pad▁｜>'] to the tokenizer
<｜▁pad▁｜>:2
Add image token = ['<image>'] to the tokenizer
<image>:128815
Added grounding-related tokens
Added chat tokens


In [2]:
prompt = apply_chat_template(processor, model.config, "document parsing.", num_images=1)
result = generate(
    model, processor, prompt, image=[PAGE], max_tokens=1024, temperature=0.0
)
print(result.text.strip()[:1200])
print(f"\n[{result.prompt_tokens} prompt tokens, {result.generation_tps:.1f} tok/s, {result.peak_memory:.2f} GB peak]")

<|det|>header [342, 12, 654, 26]<|/det|>高雄醫學大學 108 學年度行事曆 第一學期
<|det|>header [396, 27, 960, 39]<|/det|>108.04.03 107 學年度第 4 次教務會議通過 108.05.09 107 學年度第 10 次行政會議通過
<|det|>header [396, 40, 960, 52]<|/det|>108.04.03 107 學年度第 4 次教務會議通過 108.10.25 108 學年度第 1 次教務會議通過
<|det|>header [396, 53, 960, 65]<|/det|>109.02.05 108 學年度第 3 次教務會議通過 109.02.18 108 學年度第 2 次臨時教務會議通過
<|det|>header [396, 66, 960, 78]<|/det|>109.02.20 108 學年度第 7 次行政會議通過
<|det|>table [23, 85, 977, 234]<|/det|><table><tr><td colspan="7">108年8月</td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan="9"></td><td rowspan

Output arrives as `<|det|>[x1, y1, x2, y2]<|/det|>` region markers followed by the
text in each region, with coordinates normalised to 0-1000.

## Many pages at once

One literal `<image>` token covers every page, so `num_images` is the page count
while the template still emits a single placeholder. The two pages below are
unrelated — a real workflow renders a PDF to ordered page images first, but this
exercises the same path.

The `Locate <|ref|>...<|/ref|>` grounding prompts that DeepSeek-OCR accepts are
documented as experimental for this model, and they degenerate into repetition
on this page, so they are left out here.

In [3]:
PAGES = ["../images/demo_pdf1_page1.png", "../images/paper.png"]

prompt = apply_chat_template(
    processor, model.config, "Multi page parsing.", num_images=len(PAGES)
)
assert prompt.count("<image>") == 1

result = generate(
    model,
    processor,
    prompt,
    image=PAGES,
    max_tokens=1024,
    temperature=0.0,
    cropping=False,
    image_size=1024,
)
print(result.text.strip()[:1200])

<PAGE><|det|>header [350, 13, 648, 27]<|/det|>高雄醫學大學 108 學年度學月審 第一學期
<|det|>header [400, 28, 669, 40]<|/det|>108.04.03 107 學年度 第 4 次 教務會議通 108.05.09 107 學年度第 10 次 行政會議通
<|det|>header [400, 41, 669, 53]<|/det|>108.05.08 108 學年度第 1 次行政會議通
<|det|>header [400, 54, 669, 66]<|/det|>108.02.05 108 學年度第 3 次教務會議通
<|det|>header [400, 67, 669, 79]<|/det|>108.02.20 108 學年度第 3 次教務會議通
<|det|>header [400, 80, 669, 92]<|/det|>108.02.20 108 学年度第 7 次行政会议
<|det|>header [400, 93, 669, 105]<|/det|>108.02.20 108 学年度第 7 次行政会议
<|det|>header [700, 28, 960, 40]<|/det|>108.05.09 107 学年度第 10 次 行政会议
<|det|>header [700, 41, 960, 53]<|/det|>108.10.25 108 学年度第 1 次 扶督会议通过
<|det|>header [700, 54, 960, 66]<|/det|>108.02.18 108 学年度第 2 次 臨時扶督會議通過
<|det|>header [700, 67, 960, 79]<|/det|>108.02.18 108 学年度第 2 次 臨時教督會議通過
<|det|>header [700, 80, 960, 92]<|/det|>108.02.20 108 学年度第 7 次 行政會議通過
<|det|>header [700, 93, 960, 105]<|/det|>108.02.20 108 学年度第 7 次 行政會議通過
<|det|>header [700, 106, 960, 118]<|/det|>108.02.20 108 学年度第 7 次 行政會